# Complete Three-Stage Pipeline

Combines all three tasks into a single end-to-end pipeline:

```
Noisy spectrum
    → [1] CNN Denoiser          (removes Poisson noise)
    → [2] CNN Classifier        (soft mask: which elements are present)
    → [3] CNN Regressor         (predicts concentrations)
    → masked & renormalised concentrations
```

The `TwoStageRegressor` uses a **soft mask** — raw sigmoid probabilities from the
classifier instead of a hard 0/1 threshold — so a missed element is down-weighted
rather than unrecoverably zeroed out.

All three models are trained from scratch here. Pipelines compared:

| Pipeline | Flow |
|----------|------|
| **Raw** | Noisy → Regressor (noisy-trained) |
| **Denoised** | Noisy → Denoiser → Regressor (clean-trained) |
| **Complete** | Noisy → Denoiser → Soft TwoStageRegressor |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch
import sys
import os

project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.append(project_root)

from src.data.regression.generator import RegressionDataGenerator
from src.data.denoising.generator import DenoisingDataGenerator
from src.data.common.base_generator import GeneratorConfig

from src.models.regression import CNNRegressor, RegressionTrainer, evaluate_all
from src.models.regression.architectures import TwoStageRegressor
from src.models.denoising.architectures import CNNAutoencoder
from src.models.denoising.trainer import DenoisingTrainer
from src.models.classification.architectures import CNNClassifier
from src.models.classification.trainer import ClassificationTrainer

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

## Step 1 — Generate Data

All models share the same base data. Classification labels are derived directly from
regression concentrations: an element is "present" if its concentration > 0.

In [ ]:
reg_gen = RegressionDataGenerator(seed=SEED)
clean_config = GeneratorConfig.Presets.high_quality()
noisy_config = GeneratorConfig.Presets.fast_scan()

print("Generating clean data (classifier + clean regressor)...")
X_clean_train, y_clean_train = reg_gen.generate_dataset(2000, min_elements=1, max_elements=5, config=clean_config)
X_clean_val,   y_clean_val   = reg_gen.generate_dataset(400,  min_elements=1, max_elements=5, config=clean_config)

element_names    = y_clean_train.columns.tolist()
y_clean_train_np = y_clean_train.values
y_clean_val_np   = y_clean_val.values

# Binary labels for classifier: 1 if element is present, 0 otherwise
y_clf_train = (y_clean_train_np > 0).astype(np.float32)
y_clf_val   = (y_clean_val_np   > 0).astype(np.float32)

print("Generating noisy data (raw regressor)...")
X_noisy_train, y_noisy_train = reg_gen.generate_dataset(2000, min_elements=1, max_elements=5, config=noisy_config)
X_noisy_val,   y_noisy_val   = reg_gen.generate_dataset(400,  min_elements=1, max_elements=5, config=noisy_config)
y_noisy_train_np = y_noisy_train.values
y_noisy_val_np   = y_noisy_val.values

print(f"Clean: X={X_clean_train.shape}, y={y_clean_train_np.shape}")
print(f"Avg active elements per sample: {y_clf_train.sum(axis=1).mean():.2f}")

## Step 2 — Train Classifier

Multi-label CNNClassifier (41 elements, sigmoid output, BCELoss).
Trained on clean spectra so the denoised pipeline feeds it clean-like inputs.

In [ ]:
classifier = CNNClassifier(input_dim=600, num_classes=41)
clf_trainer = ClassificationTrainer(classifier, learning_rate=1e-3)

print("Training classifier...")
clf_history = clf_trainer.train(
    X_clean_train, y_clf_train,
    X_clean_val,   y_clf_val,
    epochs=60, batch_size=64, patience=10,
)

plt.figure(figsize=(10, 4))
plt.plot(clf_history["train_loss"], label="Train Loss")
plt.plot(clf_history["val_loss"],   label="Val Loss")
plt.title("Classifier — Training Curve")
plt.xlabel("Epoch"); plt.ylabel("BCE Loss")
plt.legend(); plt.grid(alpha=0.3); plt.show()

clf_metrics = clf_trainer.evaluate_metrics(X_clean_val, y_clf_val)
print("Classifier validation metrics:", clf_metrics)

## Step 3 — Train Denoiser

In [ ]:
den_gen = DenoisingDataGenerator(seed=SEED)
den_config = GeneratorConfig.Presets.fast_scan()

print("Generating denoising pairs...")
X_den_train_noisy, X_den_train_clean = den_gen.generate_dataset(2000, config=den_config)
X_den_val_noisy,   X_den_val_clean   = den_gen.generate_dataset(400,  config=den_config)

denoiser    = CNNAutoencoder(input_dim=600)
den_trainer = DenoisingTrainer(denoiser, learning_rate=1e-3)

print("Training denoiser...")
den_history = den_trainer.train(
    X_den_train_noisy, X_den_train_clean,
    X_den_val_noisy,   X_den_val_clean,
    epochs=60, batch_size=64, patience=10,
)

plt.figure(figsize=(10, 4))
plt.plot(den_history["train_loss"], label="Train Loss")
plt.plot(den_history["val_loss"],   label="Val Loss")
plt.title("Denoiser — Training Curve")
plt.xlabel("Epoch"); plt.ylabel("MSE Loss")
plt.legend(); plt.grid(alpha=0.3); plt.show()

## Step 4 — Train Regressors

- **Clean regressor**: trained on `high_quality` spectra — used in the denoised and complete pipelines.
- **Raw regressor**: trained on `fast_scan` noisy spectra — the baseline.

In [ ]:
cnn_clean   = CNNRegressor()
trainer_clean = RegressionTrainer(cnn_clean, learning_rate=1e-3)
print("Training clean regressor...")
history_clean = trainer_clean.train(
    X_clean_train, y_clean_train_np,
    X_clean_val,   y_clean_val_np,
    epochs=60, batch_size=64, patience=10,
)

cnn_noisy   = CNNRegressor()
trainer_noisy = RegressionTrainer(cnn_noisy, learning_rate=1e-3)
print("Training raw (noisy) regressor...")
history_noisy = trainer_noisy.train(
    X_noisy_train, y_noisy_train_np,
    X_noisy_val,   y_noisy_val_np,
    epochs=60, batch_size=64, patience=10,
)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, hist, title in zip(axes,
                            [history_clean, history_noisy],
                            ["Clean Regressor", "Raw Regressor"]):
    ax.plot(hist["train_loss"], label="Train")
    ax.plot(hist["val_loss"],   label="Val")
    ax.set_title(f"{title} — Training Curve")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
    ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Step 5 — Assemble Complete Pipeline

The `TwoStageRegressor` is initialised with `soft_mask=True` (default): the classifier's
sigmoid probabilities are used as continuous weights instead of a hard 0/1 threshold.

In [ ]:
# soft_mask=True: probs used directly as weights (no hard zeroing)
two_stage = TwoStageRegressor(
    classifier=clf_trainer.model,
    regressor=trainer_clean.model,
    soft_mask=True,
)
two_stage.eval()
print("TwoStageRegressor assembled (soft mask).")

# Helper: run TwoStageRegressor on a numpy array
def predict_two_stage(model, X, batch_size=64):
    device = next(model.parameters()).device
    preds = []
    for i in range(0, len(X), batch_size):
        batch = torch.FloatTensor(X[i:i+batch_size]).to(device)
        with torch.no_grad():
            preds.append(model(batch).cpu().numpy())
    return np.concatenate(preds, axis=0)

## Step 6 — Generate Test Set & Evaluate

In [ ]:
print("Generating noisy test set...")
X_test_noisy, y_test = reg_gen.generate_dataset(400, min_elements=1, max_elements=5, config=noisy_config)
y_test_np = y_test.values

X_test_denoised = den_trainer.predict(X_test_noisy)
print(f"Test set: {X_test_noisy.shape}")

# --- Evaluate all three pipelines ---
metrics_raw      = trainer_noisy.evaluate(X_test_noisy,    y_test_np, element_names=element_names)
metrics_denoised = trainer_clean.evaluate(X_test_denoised, y_test_np, element_names=element_names)

preds_complete   = predict_two_stage(two_stage, X_test_denoised)
metrics_complete = evaluate_all(y_test_np, preds_complete)

summary = pd.DataFrame({
    "Raw":      {k: v for k, v in metrics_raw.items()      if k != "per_element_mae"},
    "Denoised": {k: v for k, v in metrics_denoised.items() if k != "per_element_mae"},
    "Complete": {k: v for k, v in metrics_complete.items() if k != "per_element_mae"},
}).T

print("\n=== Pipeline Comparison ===")
print(summary.to_string(float_format="{:.4f}".format))

## Step 7 — Per-Element MAE Comparison

In [ ]:
per_el_raw      = metrics_raw["per_element_mae"]
per_el_denoised = metrics_denoised["per_element_mae"]
per_el_complete = metrics_complete["per_element_mae"]

# Align and sort by raw MAE (descending — hardest elements first)
per_el_df = pd.DataFrame({
    "Raw":      per_el_raw,
    "Denoised": per_el_denoised,
    "Complete": per_el_complete,
}).dropna().sort_values("Raw", ascending=False)

per_el_df.head(20).plot(kind="bar", figsize=(16, 5), width=0.7,
                         color=["tomato", "steelblue", "seagreen"])
plt.title("Per-Element MAE — Top 20 Hardest Elements")
plt.xlabel("Element"); plt.ylabel("Masked MAE")
plt.xticks(rotation=45, ha="right")
plt.legend(); plt.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.show()

## Step 8 — Visual Sample Comparison

For a few test samples: spectrum (noisy vs denoised) and concentration predictions
from all three pipelines.

In [ ]:
preds_raw      = trainer_noisy.predict(X_test_noisy)
preds_denoised = trainer_clean.predict(X_test_denoised)
energies       = np.arange(0, 30, 0.05)

fig, axes = plt.subplots(3, 2, figsize=(16, 13))

for i in range(3):
    ax_spec = axes[i, 0]
    ax_bar  = axes[i, 1]

    ax_spec.plot(energies, X_test_noisy[i],    label="Noisy",    alpha=0.5, color="red",       lw=1)
    ax_spec.plot(energies, X_test_denoised[i], label="Denoised", alpha=0.9, color="steelblue", lw=1.5)
    ax_spec.set_title(f"Sample {i} — Spectrum")
    ax_spec.set_xlabel("Energy (keV)"); ax_spec.set_ylabel("Intensity")
    ax_spec.legend(); ax_spec.grid(alpha=0.3)

    active_mask = y_test_np[i] > 0
    active_els  = [element_names[j] for j in range(41) if active_mask[j]]
    true_vals   = y_test_np[i][active_mask]
    raw_vals    = preds_raw[i][active_mask]
    den_vals    = preds_denoised[i][active_mask]
    cpl_vals    = preds_complete[i][active_mask]

    x_pos = np.arange(len(active_els))
    w = 0.2
    ax_bar.bar(x_pos - 1.5*w, true_vals, w, label="True",     color="steelblue", alpha=0.85)
    ax_bar.bar(x_pos - 0.5*w, raw_vals,  w, label="Raw",      color="tomato",    alpha=0.85)
    ax_bar.bar(x_pos + 0.5*w, den_vals,  w, label="Denoised", color="goldenrod", alpha=0.85)
    ax_bar.bar(x_pos + 1.5*w, cpl_vals,  w, label="Complete", color="seagreen",  alpha=0.85)
    ax_bar.set_xticks(x_pos)
    ax_bar.set_xticklabels(active_els)
    ax_bar.set_ylabel("Relative Concentration")
    ax_bar.set_title(f"Sample {i} — Concentrations")
    ax_bar.legend(); ax_bar.grid(axis="y", alpha=0.3)

plt.tight_layout(); plt.show()

## Step 9 — Noise Ablation (all three pipelines)

In [ ]:
noise_levels = [500, 1000, 3000, 5000, 10000, 20000, 30000]
mae_raw_list, mae_den_list, mae_cpl_list = [], [], []

for n in noise_levels:
    cfg = GeneratorConfig.Presets.fast_scan()
    cfg.n_counts_range = (n, n)

    X_n, y_n = reg_gen.generate_dataset(200, min_elements=1, max_elements=5, config=cfg)
    y_n_np   = y_n.values
    X_n_den  = den_trainer.predict(X_n)

    m_raw = trainer_noisy.evaluate(X_n,     y_n_np)
    m_den = trainer_clean.evaluate(X_n_den, y_n_np)
    p_cpl = predict_two_stage(two_stage, X_n_den)
    m_cpl = evaluate_all(y_n_np, p_cpl)

    mae_raw_list.append(m_raw["masked_mae"])
    mae_den_list.append(m_den["masked_mae"])
    mae_cpl_list.append(m_cpl["masked_mae"])
    print(f"n={n:6d}  raw={m_raw['masked_mae']:.4f}  den={m_den['masked_mae']:.4f}  cpl={m_cpl['masked_mae']:.4f}")

plt.figure(figsize=(10, 5))
plt.plot(noise_levels, mae_raw_list, marker="o", label="Raw",      color="tomato")
plt.plot(noise_levels, mae_den_list, marker="s", label="Denoised", color="steelblue")
plt.plot(noise_levels, mae_cpl_list, marker="^", label="Complete", color="seagreen")
plt.xscale("log")
plt.xlabel("Noise counts (n_counts)")
plt.ylabel("Masked MAE (active elements)")
plt.title("MAE vs Noise Level — All Pipelines")
plt.legend(); plt.grid(alpha=0.3); plt.show()